# Product Listing Generator
Automatically generates e-commerce product listings from images using GPT-4o vision.

## Setup — install dependencies

In [1]:
# Run this cell first — installs everything you need
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "openai", "python-dotenv", "requests", "Pillow", "--quiet"])
print("All packages installed.")

All packages installed.


## Cell 1 — load API key & initialise client

In [2]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(".env")
api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    print("API key loaded successfully.")
    print(f"Key starts with: {api_key[:7]}...")
else:
    raise ValueError("API key not found. Make sure your .env file contains: OPENAI_API_KEY=sk-...")

client = OpenAI(api_key=api_key)
print("OpenAI client ready.")

API key loaded successfully.
Key starts with: sk-svca...
OpenAI client ready.


## Cell 2 — download sample product images automatically
This cell creates the `product_images/` folder and downloads 3 real product photos from the web.  
**You do not need to do anything manually — just run it.**

In [5]:
import requests
import os
from PIL import Image
from io import BytesIO

os.makedirs("product_images", exist_ok=True)

# Free-to-use product photos from Unsplash (no login needed)
image_sources = [
    {
        "filename": "product_images/p1.jpg",
        "url": "https://images.unsplash.com/photo-1505740420928-5e560c06d30e?w=600&q=80",
        "description": "Headphones"
    },
    {
        "filename": "product_images/p2.jpg",
        "url": "https://images.unsplash.com/photo-1542291026-7eec264c27ff?w=600&q=80",
        "description": "Running shoe"
    },
    {
        "filename": "product_images/p3.jpg",
        "url": "https://images.unsplash.com/photo-1602143407151-7111542de6e8?w=600&q=80",
        "description": "Water bottle"
    }
]

for item in image_sources:
    if os.path.exists(item["filename"]):
        print(f"  Already exists: {item['filename']} — skipping")
        continue
    try:
        resp = requests.get(item["url"], timeout=15)
        resp.raise_for_status()
        img = Image.open(BytesIO(resp.content)).convert("RGB")
        img.save(item["filename"], "JPEG", quality=85)
        size_kb = os.path.getsize(item["filename"]) // 1024
        print(f"  Downloaded: {item['filename']}  ({item['description']}, {size_kb} KB)")
    except Exception as e:
        print(f"  Could not download {item['filename']}: {e}")

print("\nAll images ready:")
for f in sorted(os.listdir("product_images")):
    size_kb = os.path.getsize(f"product_images/{f}") // 1024
    print(f"  {f}  ({size_kb} KB)")

  Already exists: product_images/p1.jpg — skipping
  Already exists: product_images/p2.jpg — skipping
  Already exists: product_images/p3.jpg — skipping

All images ready:
  p1.jpg  (21 KB)
  p2.jpg  (25 KB)
  p3.jpg  (19 KB)


## Cell 3 — helper functions

In [ ]:
# Your school key may only allow certain models.
# The code will automatically try each one until it finds one that works.
MODELS_TO_TRY = [
    "gpt-4o-mini",
    "gpt-4-turbo",
    "gpt-4-vision-preview",
    "gpt-3.5-turbo",
]

def find_working_model(test_models):
    """Try each model and return the first one the key can access."""
    for m in test_models:
        try:
            client.chat.completions.create(
                model=m, max_tokens=5,
                messages=[{"role":"user","content":"hi"}]
            )
            print(f"  Model available: {m}")
            return m
        except Exception as e:
            print(f"  Model not available: {m}")
    raise RuntimeError("None of the models are accessible with this API key.")

MODEL = find_working_model(MODELS_TO_TRY)
print(f"Using model: {MODEL}")

import base64
import json

def encode_image(path: str) -> str:
    """Read an image file and return it as a base64-encoded string."""
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

def make_prompt(name: str, price: float, category: str) -> str:
    """Build the instruction prompt with product metadata."""
    return f"""You are an expert e-commerce copywriter.
Analyse the product image and write a compelling listing for:
- Name: {name}
- Price: ${price:.2f}
- Category: {category}

Return ONLY valid JSON — no markdown fences, no extra text — in this exact format:
{{
  "title": "SEO-friendly title, max 60 characters",
  "description": "Persuasive 150-200 word description highlighting benefits",
  "features": ["Feature 1", "Feature 2", "Feature 3", "Feature 4", "Feature 5"],
  "keywords": "comma-separated list of 10-15 relevant SEO keywords"
}}"""

def generate_listing(image_path: str, name: str, price: float, category: str) -> dict:
    """Send image + prompt to GPT-4o and return a parsed listing dict."""
    img_b64  = encode_image(image_path)
    prompt   = make_prompt(name, price, category)
    response = client.chat.completions.create(
        model=MODEL,
        max_tokens=1000,
        messages=[{
            "role": "user",
            "content": [
                {"type": "text",      "text": prompt},
                {"type": "image_url", "image_url": {
                    "url": f"data:image/jpeg;base64,{img_b64}"
                }}
            ]
        }]
    )
    raw   = response.choices[0].message.content
    clean = raw.strip().replace("```json", "").replace("```", "")
    return json.loads(clean)

print("Helper functions defined.")

SyntaxError: unterminated f-string literal (detected at line 25) (129180868.py, line 25)

## Cell 4 — generate listings for all 3 products

In [7]:
products = [
    {"image": "product_images/p1.jpg", "name": "Wireless Headphones",   "price": 79.99,  "category": "Electronics"},
    {"image": "product_images/p2.jpg", "name": "Running Shoes",         "price": 129.99, "category": "Footwear"},
    {"image": "product_images/p3.jpg", "name": "Stainless Water Bottle", "price": 34.99,  "category": "Kitchen"},
]

results = []

for p in products:
    if not os.path.exists(p["image"]):
        print(f"  SKIPPED — image not found: {p['image']}  (re-run Cell 2 first)")
        continue
    try:
        print(f"  Generating: {p['name']} ...", end=" ", flush=True)
        listing = generate_listing(p["image"], p["name"], p["price"], p["category"])
        results.append({"product": p["name"], "listing": listing})
        print("Done")
    except json.JSONDecodeError as e:
        print(f"JSON parse error — {e}")
    except Exception as e:
        print(f"Error — {e}")

with open("listings.json", "w") as f:
    json.dump(results, f, indent=2)

print(f"\n{len(results)}/3 listing(s) saved to listings.json")

  Generating: Wireless Headphones ... 

NameError: name 'json' is not defined

## Cell 5 — preview all results

In [8]:
if not results:
    print("No results yet. Run cells in order: Setup → 1 → 2 → 3 → 4, then come back here.")
else:
    for r in results:
        l = r["listing"]
        print("=" * 60)
        print(f"Product  : {r['product']}")
        print(f"Title    : {l.get('title', 'N/A')}")
        print(f"Keywords : {l.get('keywords', 'N/A')}")
        print(f"\nDescription:\n{l.get('description', 'N/A')}")
        print("\nFeatures:")
        for feat in l.get("features", []):
            print(f"  - {feat}")
        print()

No results yet. Run cells in order: Setup → 1 → 2 → 3 → 4, then come back here.
